# Agents

Các tác nhân (agent) kết hợp mô hình ngôn ngữ với các công cụ để tạo ra những hệ thống có thể suy luận về nhiệm vụ, quyết định nên sử dụng công cụ nào và làm việc theo từng bước để tiến tới lời giải.

Một LLM Agent vận hành các công cụ theo vòng lặp nhằm đạt được mục tiêu. Tác nhân sẽ tiếp tục chạy cho đến khi đạt điều kiện dừng — ví dụ như khi mô hình tạo ra kết quả cuối cùng hoặc khi chạm đến giới hạn số vòng lặp.

![Agent](./images/agent.png)

# Tools 
Các công cụ (tools) trao cho agent khả năng thực hiện hành động. Agent vượt xa việc chỉ đơn thuần “gắn” mô hình với công cụ bằng cách hỗ trợ:
- Gọi nhiều công cụ theo chuỗi (được kích hoạt từ một prompt duy nhất)
- Gọi công cụ song song khi phù hợp
- Lựa chọn công cụ một cách động dựa trên các kết quả trước đó
- Cơ chế thử lại công cụ và xử lý lỗi

Duy trì trạng thái xuyên suốt các lần gọi công cụ

## Defining tools
Pass a list of tools to the agent.

In [1]:
from langchain.tools import tool
from langchain.agents import create_agent

### Search Tool 
https://docs.langchain.com/oss/python/integrations/tools


In [2]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

search.invoke("Rikkeisoft?")

'In 2012, Rikkeisoft was established by six software developers on a lofty mission: to drive technological innovation and deliver lasting values. Gain a comprehensive roadmap for your digital evolution. Rikkeisoft ’s experts analyze your processes, identify bottlenecks, and propose tailored digital solutions. Hiện tại, Rikkeisoft đang đặt mục tiêu tham gia vào các thị trường quốc tế mới. rikkeisoft .com и ещё 2 ссылки. Rikkeisoft Corporation has 66 repositories available. Follow their code on GitHub.Covid-19 Maps - Training Project. rikkeisoft /covid19-map’s past year of commit activity. Apart from traditional IT services, Rikkeisoft also provides a plethora of breakthrough technologies via its ecosystem of subsidiaries, comprising of: Rikkei Digital: Digital conversion...'

In [3]:
from langchain_community.tools import DuckDuckGoSearchResults
search = DuckDuckGoSearchResults(output_format="list")

search.invoke("Rikkeisoft?")

[{'snippet': 'In 2012, Rikkeisoft was established by six software developers on a lofty mission: to drive technological innovation and deliver lasting values. Following in the footsteps of our founders, we cherish the tradition of excellence.',
  'title': 'Rikkeisoft - Trusted IT Outsourcing Provider',
  'link': 'https://rikkeisoft.com/'},
 {'snippet': "Rikkeisoft tự hào thông báo: Rikkei Japan - pháp nhân của Rikkeisoft tại Nhật Bản đã chính thức được công nhận là “Great Place to Work 2025”, danh hiệu uy tín toàn cầu dành cho các tổ chức có môi trường làm việc xuất sắc! Lần đầu tiên có mặt trong danh sách này, Rikkei Japan Leading Software solutions and Services provider in Vietnam. Founded in 2012, Rikkeisoft is an award-winning, leading Vietnamese IT Enterprise. We help our partner companies grow sustainably with... Established in 2012, Rikkeisoft is a leading provider of technology resources & services for the US, Europe, and Asia-Pacific (APAC) region. For 10 years+, we have been 

In [4]:
@tool(description="Search for information about Rikkeisoft.")
def search_rikkeisoft_information(query: str) -> str:
    """Search for information about Rikkeisoft."""
    from langchain_community.tools import DuckDuckGoSearchRun
    search = DuckDuckGoSearchRun()
    result = search.invoke(query)
    if result:
        return result
    else:
        return "No information found"

search_rikkeisoft_information.invoke({
    "query": "Rikkeisoft?"
})





'In 2012, Rikkeisoft was established by six software developers on a lofty mission: to drive technological innovation and deliver lasting values. Rikkeisoft ’ s strength lies in developing web and mobile app systems because these are the first two areas of activity since Rikkeisoft ... FREE RESOURCE Ultimate Guide to Successful Salesforce Implementation Avoid Salesforce implementation failures! This eBook from Rikkeisoft guides you ... On the afternoon of July 17, Rikkeisoft held its 10th Anniversary Celebration, gathering 1,500 employees, partners, international customers and ... At Rikkeisoft , we provide our full assistance to developers as they learn and obtain Japanese language and IT-related certifications so that they can ...'

### Get current time tool

In [5]:
@tool(description="Get current time by UTC offset.")
def get_current_time(utc: int = 0) -> str:
    """Get current time by UTC offset."""
    from datetime import datetime, timedelta, timezone
    tz = timezone(timedelta(hours=utc))
    return datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")


In [6]:

@tool
def calculate_years_of_establishment(start_year: int) -> str:
    """Calculate the number of years since the establishment of Rikkeisoft."""
    from datetime import datetime
    return f"Rikkeisoft was established in {datetime.now().year - start_year} years ago"

calculate_years_of_establishment.invoke({
    "start_year": 2010
})

'Rikkeisoft was established in 16 years ago'

## Model
The model is the reasoning engine of your agent. It can be specified in multiple ways, supporting both static and dynamic model selection.

### Static model
Static models are configured once when creating the agent and remain unchanged throughout execution. This is the most common and straightforward approach.

In [7]:
from config import settings
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model=settings.LLM_CHAT_MODEL,api_key=settings.LLM_API_KEY)

agent = create_agent(model=model, tools=[
    search_rikkeisoft_information,
    get_current_time,
    calculate_years_of_establishment
])


In [8]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the current time in Vietnam?"}]}
)

In [9]:
result

{'messages': [HumanMessage(content='what is the current time in Vietnam?', additional_kwargs={}, response_metadata={}, id='a182eb86-775d-4742-969b-badd8e84805e'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{"utc": 7}'}, '__gemini_function_call_thought_signatures__': {'67525a66-c77d-4f45-98e5-1a79d1a9a22a': 'EuMBCuABAb4+9vsjYuN86hQ3z6dmUsviDfyMQgHyekvl/5Ez/o3nNoMtYk/4KHUPEWNjpLR4L0zGn0crIWE1Fb3R7P+uMM5Eek/21g/NNK6GV3W7QsfMCl87Spqc1tZ+mmuDYdpYhcu4ECDssqdl1vmP5QOoUlHnaimr7BJGZOeY9UAYTk+OHBhASlw75qTyOaeqNNUKYvfJYf++HDNVWXz+QsjzgUdULhVtR3o4QEQmSNgNT4giD24nPgCFCICI/s+N5kAXOQQxbQ447b4WlpvJlM22bVFMxe14a/B98OYqYxq/B+8='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d2560-9e51-7c32-9691-ffa6ab7da052-0', tool_calls=[{'name': 'get_current_time', 'args': {'utc': 7}, 'id': '67525a66-c77d-4f45-98e5-1a79d1a9a22a', 'type': 'tool_cal

In [10]:
final_message = result["messages"][-1].content
print(final_message)

[{'type': 'text', 'text': 'The current time in Vietnam is 21:23:03 on March 25, 2026.'}]


In [11]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Rikkeisoft được thành lập bao nhiêu năm rồi?"}]}
)
result
final_message = result["messages"][-1].content
print(final_message)


[{'type': 'text', 'text': 'Rikkeisoft được thành lập vào ngày 6 tháng 4 năm 2012. Tính đến hiện tại (năm 2024), Rikkeisoft đã hoạt động được **12 năm**. \n\n(Lưu ý: Nếu tính đến thời điểm năm 2026 như trong dữ liệu hệ thống vừa cập nhật, con số sẽ là 14 năm).', 'extras': {'signature': 'ErYBCrMBAb4+9vuJ7TRw2B6vKmYU/RSBo5zn8FWxsseGXm5CKjv5nU2wWrYZITfbRkxyypr/Fz7Ya5vyLcNdUTGyI8WJbQL9xHM3G/uSz9THGglLpK2U4ADPRqgclrH2unQ54xUcWEAYoNbRE7yZkVXnXB+9UM4g97w+jY47R2mQSlesD/V0maY3M/oKSJIgqlfF3aeFeJQgq4NMgzRpbyap6T3kPRnW3s/qPmYIdlMNvtklkxdYGlA='}}]


In [12]:
result

{'messages': [HumanMessage(content='Rikkeisoft được thành lập bao nhiêu năm rồi?', additional_kwargs={}, response_metadata={}, id='1638779e-a0ad-4935-9537-4b802addcd80'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_rikkeisoft_information', 'arguments': '{"query": "n\\u0103m th\\u00e0nh l\\u1eadp Rikkeisoft"}'}, '__gemini_function_call_thought_signatures__': {'496f57c8-603b-4143-aa3f-d565b8e951fc': 'ErcCCrQCAb4+9vvNcW94JjRYl+z1UY1ivFPnwHyduCUDOmJ1+6knI28IyH5W8z9fUctgxkqC6TYV0Lb73watRlLCK4UeEPwHG2qIwMO47PP/zEBQBPqImjAPkNL0GYMhFtSMpTNR8gp0fUcJb8U+LshvE20FQxKVCzf4+iaY/xVhe6nK+mPUDvwMCC4eVS8KvW24+ehkU5Epxx7ceg4KoXhgLPa+xw/BnHhjtqwTfbhTb7wXnt8dBhDUByLeqoNxhuQ9uFk10XB0lD641Vnf/FxSyft6IcjxmH8ssXQQTgLlFuiMdFducGQWerr6xLE1SrDDdjTWedtFtliNn9J2zu6T6OfOtsBsLIrE+6c/6gB/7mivyAfNoQ44AToIcWmDSDATz82fwy0c6vRqgKu70bhGaxN4RXajB80='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'},

### Dynamic Model là gì?

**Dynamic model** là cơ chế cho phép **chọn mô hình LLM tại thời điểm runtime** dựa trên **ngữ cảnh, trạng thái hiện tại hoặc logic tùy biến** (ví dụ: độ phức tạp câu hỏi, chi phí, độ trễ, người dùng trả phí hay miễn phí).  
Thay vì cố định một model (như GPT-4 hoặc Gemini-Pro), hệ thống có thể **tự động chuyển đổi model** để đạt hiệu quả tốt nhất giữa **chất lượng – chi phí – tốc độ**.

---

### Vì sao cần Dynamic Model?

Dynamic model giúp:
- **Routing thông minh**: câu hỏi đơn giản dùng model rẻ/nhanh, câu hỏi phức tạp dùng model mạnh.
-  **Tối ưu chi phí**: giảm dùng model đắt khi không cần thiết.
-  **Cải thiện hiệu năng**: ưu tiên model phản hồi nhanh trong các tình huống realtime.
- **Linh hoạt mở rộng**: dễ thêm model mới mà không thay đổi toàn bộ hệ thống.

### Middleware với `@wrap_model_call`


Để dùng dynamic model, bạn tạo middleware bằng decorator `@wrap_model_call`. Middleware này sẽ **chỉnh sửa model trong request** trước khi LLM được gọi.


In [17]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse


basic_model = ChatGoogleGenerativeAI(api_key=settings.LLM_API_KEY,model="gemini-2.5-flash")
advanced_model = ChatGoogleGenerativeAI(api_key=settings.LLM_API_KEY,model="gemini-2.5-pro")

@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """Choose model based on conversation complexity."""
    message_count = len(request.state["messages"])

    if message_count > 10:
        model = advanced_model
    else:
        model = basic_model

    return handler(request.override(model=model))

agent = create_agent(
    model=basic_model,  # Default model
    tools=[search_rikkeisoft_information,get_current_time,calculate_years_of_establishment],
    middleware=[dynamic_model_selection]
)

In [18]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Rikkeisoft được thành lập bao nhiêu năm rồi?"}]}
)
result
final_message = result["messages"][-1].content
print(final_message)

[{'type': 'text', 'text': 'Rikkeisoft được thành lập cách đây 14 năm.', 'extras': {'signature': 'Ct0BAb4+9vs0vpT3mcyU/B3fOwzMHu5jKPSbMdkatDoX2ifhLuSfk5DoY7CaVclJQE+kjXLiUJV4aFDIWQtXctdSs1e3CaoviMDmWVnQ9jfJaeCN3NYggWo9DUcA20bzUqP+L6TJ7KH2oXCQfBWuJUhFJpTSApnG+6dVI2xAFxtz+f16NcUK4zMiYYodWU90W6qWk0WrxIeSwbjkvnL7OYAWNIJDMbqky7TzqgfsWtsKfQtaKxNS5jJZqFGX8mCtlQv4GR83OlzDJPqHjs9w7gFvePT1VCJ/TosSTX10QNE='}}]


### System Prompt 
**System prompt** là thông điệp dùng để **định hình hành vi, vai trò và phong cách làm việc của agent** ngay từ đầu.  
Nó trả lời cho câu hỏi: *“Agent này nên suy nghĩ và phản hồi như thế nào?”*
Trong LangChain, có thể truyền system prompt khi tạo agent để kiểm soát:
- Cách agent tiếp cận nhiệm vụ
- Mức độ chi tiết / ngắn gọn
- Tính cách, vai trò (assistant, chuyên gia, reviewer, v.v.)

In [19]:
agent = create_agent(
    model=model,  # Default model
    tools=[search_rikkeisoft_information,get_current_time,calculate_years_of_establishment],
    middleware=[dynamic_model_selection],
    system_prompt="You are a helpful assistant that can answer questions and help with tasks."
)

In [20]:
agent.invoke(
    {"messages": [{"role": "user", "content": "what is the current time?"}]}
)

{'messages': [HumanMessage(content='what is the current time?', additional_kwargs={}, response_metadata={}, id='acc054d0-bc6f-4b22-8853-e64f39b1fd77'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'e937ad22-d1db-4c18-a51f-b3235a7dacee': 'CoUCAb4+9vsH17/suzhl9MLQVSEMlcbD1/Apc5EAmJ3vGXMWQhFq+qR/4sGJPtt4rOhPLFR2xmu8vpr9OHqNQ/lkUHDqcMVZjdZY/17ZZGgYI2zNGNSHmbWqz9psn3x4YEeo6s69EMjFSN6YdjJf99K/U2nRxPOhP8G5KJOSURuTTgo1dvRqEOyQqyZbC5hLVky1gm5u4WJ//4dIBSAR6eC3fTd/ZJzdbH6teyMJVnLBZdzPfBHWpf69AfB08CYU/SNYFHQuznatbpvypGhANQ4zJXE/BIVwaZ8au9Tlfp7j7Jz+Am4ErCmrXBD3y2d1U5qFKV7flieBhJcDZjSgcEBEpFVtO0WK'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d2563-4013-7730-bf79-3327427da813-0', tool_calls=[{'name': 'get_current_time', 'args': {}, 'id': 'e937ad22-d1db-4c18-a51f-b3235a7dacee', 'type'

### Memory

Trong LangChain, **Agent tự động duy trì lịch sử hội thoại** thông qua *message state*.  
Phần thông tin này có thể xem như **short-term memory** (bộ nhớ ngắn hạn) của agent, giúp agent:
- Hiểu ngữ cảnh cuộc trò chuyện
- Trả lời nhất quán qua nhiều lượt
- Tham chiếu lại thông tin đã nói trước đó


In [25]:
from langgraph.checkpoint.memory import MemorySaver  
checkpointer = MemorySaver()  # In-memory 
from typing import Any
agent = create_agent(
    model,
    tools=[search_rikkeisoft_information,get_current_time,calculate_years_of_establishment],
    checkpointer=checkpointer
)
config = {"configurable": {"thread_id": "session_1"}}
result = agent.invoke({
    "messages": [{"role": "user", "content": "Rikkeisoft được thành lập bao nhiêu năm rồi?"}],
}, config)

In [26]:
print(result["messages"][-1].content)

[{'type': 'text', 'text': 'Rikkeisoft được thành lập vào năm **2012**. Tính đến năm 2026, Rikkeisoft đã có **14 năm** hình thành và phát triển.', 'extras': {'signature': 'Es8BCswBAb4+9vvi1eclsvYdisAf+rczu1MK35BqMWLCVEOcQvU/V2db5S9LNhKa4+bA0LDA0fQsvb42q58JzKa8wmFhowkBJEwQOFDWPHsibvZdqCno8/FejHFv6kzlIYCz32TFSdTMnkorl551n9+H6QIg/SBwcQFRdLHqtWmsAp6iMF2RrZgUeluVEbuRTK8nBlGRg36jAyj2TORUMmkvi/DxKZdY8fSxCyL97cU8I1Rxkrt/vB6t75QLnqt04mQtwJMWv/h505rY9uS903fG'}}]


In [27]:
#print checkpointer
checkpointer.get(config)

{'v': 4,
 'ts': '2026-03-24T15:50:18.422591+00:00',
 'id': '1f127992-8441-647e-8007-47a8c7180b23',
 'channel_versions': {'__start__': '00000000000000000000000000000002.0.25347472755411415',
  'messages': '00000000000000000000000000000009.0.6499747896018543',
  'branch:to:model': '00000000000000000000000000000009.0.6499747896018543',
  '__pregel_tasks': '00000000000000000000000000000008.0.4719708250652065'},
 'versions_seen': {'__input__': {},
  '__start__': {'__start__': '00000000000000000000000000000001.0.9965913552319414'},
  'model': {'branch:to:model': '00000000000000000000000000000008.0.4719708250652065'},
  'tools': {}},
 'updated_channels': ['messages'],
 'channel_values': {'messages': [HumanMessage(content='Rikkeisoft được thành lập bao nhiêu năm rồi?', additional_kwargs={}, response_metadata={}, id='06daca9d-7942-416d-a818-bb46314db35d'),
   AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_rikkeisoft_information', 'arguments': '{"query": "n\\u0103m th

In [28]:
result_2 = agent.invoke({
    "messages": [{"role": "user", "content": "Tôi đã hỏi bạn gì nhỉ?"}],
}, config)
print(result_2["messages"][-1].content)
checkpointer.get(config)


[{'type': 'text', 'text': 'Bạn vừa hỏi tôi là: **"Rikkeisoft được thành lập bao nhiêu năm rồi?"**\n\nTôi đã trả lời rằng Rikkeisoft được thành lập vào năm 2012, và tính đến năm 2026 thì công ty đã hoạt động được 14 năm.', 'extras': {'signature': 'EusCCugCAb4+9vum2SLCNPqKWHNS4wArlY40/vjWucoJ8Wt3NrrnhG5B1EHQz+IDwYlBfZGhaPToK+lR6iPLNizJ6iw0Jg8gIP94hXddIvWyIGHAgNK03Rt0Fqq1oXbgxsbICrfFw9WQeZmS5J5mzBiiYFirK9thHuyIx75matcYeGQ+RIKbmAD2VfsJAqLPGJ/rtAQUyf4E89cNJD0yaXkd3cGtdq6FqG55ACmDnX0cfqBAWFs0yeBE0GnTCgBeUFfiSInT9+7b1peppFaU2FiZT0xyciCT4unDUmOrLagRN0G+QztSYDVRdxztLP1XX19ZqZcCe9KVmYNupfv2yaqcomAlMXHgrbpsJRGqVys9xGnQBV77xyvukrYsoyyoxOUdX0P2PXx49wdUmuQ2lWcpmPk2vuBJrrJ2FJ35s/wXSBEYkUlFXpTkX7PfU2F7agCrc/A7RWV13QvNtXcibsRWGeRaeOVXpAt+mqgv'}}]


{'v': 4,
 'ts': '2026-03-24T15:50:20.332885+00:00',
 'id': '1f127992-9679-6155-800a-ff8cfc420aaa',
 'channel_versions': {'__start__': '00000000000000000000000000000011.0.32343245238201324',
  'messages': '00000000000000000000000000000012.0.46535829760770997',
  'branch:to:model': '00000000000000000000000000000012.0.46535829760770997',
  '__pregel_tasks': '00000000000000000000000000000008.0.4719708250652065'},
 'versions_seen': {'__input__': {},
  '__start__': {'__start__': '00000000000000000000000000000010.0.28317838124212624'},
  'model': {'branch:to:model': '00000000000000000000000000000011.0.32343245238201324'},
  'tools': {}},
 'updated_channels': ['messages'],
 'channel_values': {'messages': [HumanMessage(content='Rikkeisoft được thành lập bao nhiêu năm rồi?', additional_kwargs={}, response_metadata={}, id='06daca9d-7942-416d-a818-bb46314db35d'),
   AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_rikkeisoft_information', 'arguments': '{"query": "n\\u0103

In [29]:
for chunk in agent.stream({
    "messages": [{"role": "user", "content": "Bây giờ là mấy giờ Nhật?"}]
}, config, stream_mode="values"):
    # Each chunk contains the full state at that point
    latest_message = chunk["messages"][-1]
    if latest_message.content:
        print(f"Agent: {latest_message.content}")
    elif latest_message.tool_calls:
        print(f"Calling tools: {[tc['name'] for tc in latest_message.tool_calls]}")

Agent: Bây giờ là mấy giờ Nhật?
Calling tools: ['get_current_time']
Agent: 2026-03-25 00:50:24
Agent: [{'type': 'text', 'text': 'Bây giờ tại Nhật Bản là **00:50**, thứ Tư ngày 25 tháng 03 năm 2026.'}]
